<a href="https://colab.research.google.com/github/smartanilmali234-art/Anilmali/blob/main/Titanic_Model_Evaluation_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 - Task 1: Model Evaluation & Tuning: Beyond Accuracy

## Objective
The objective of this project is to evaluate a machine learning classification model using multiple performance metrics instead of relying only on accuracy. The project also demonstrates how hyperparameter tuning can improve model performance using GridSearchCV.

## Dataset
- Dataset: Titanic Dataset
- Problem Type: Classification
- Target Variable: Survived

## Tasks Performed
- Loaded and preprocessed the Titanic dataset.
- Trained a baseline Decision Tree Classifier.
- Evaluated the model using Accuracy, Precision, Recall, and F1-Score.
- Explained why accuracy alone can be misleading for imbalanced datasets.
- Used GridSearchCV to tune the `max_depth` and `min_samples_split` hyperparameters.
- Compared the performance of the original and tuned models.
- Summarized the results and key learnings.

## Tools & Libraries
- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib (optional)
- Google Colab

## Expected Outcome
By the end of this project, we will understand how to properly evaluate classification models and improve their performance through hyperparameter tuning instead of relying only on default model settings.

In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [23]:
df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [25]:
df = df.drop(columns=["Cabin","Name","Ticket"])

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df = pd.get_dummies(df, columns=["Sex","Embarked"], drop_first=True)

In [26]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [28]:
model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [29]:
pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)

Accuracy: 0.7206703910614525


In [30]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.77      0.75      0.76       105
           1       0.66      0.68      0.67        74

    accuracy                           0.72       179
   macro avg       0.71      0.71      0.71       179
weighted avg       0.72      0.72      0.72       179



In [31]:
print(confusion_matrix(y_test, pred))

[[79 26]
 [24 50]]


Accuracy measures the percentage of correct predictions.

However, if one class is much larger than another, a model can predict only the majority class and still achieve high accuracy.

Precision tells us how many predicted positives were actually correct.

Recall tells us how many actual positives were correctly found.

F1-score balances Precision and Recall.

Therefore, Precision, Recall, and F1-score provide a more reliable evaluation than accuracy alone, especially for imbalanced datasets.

In [32]:
params = {
    "max_depth":[2,3,4,5,6,7,8,10],
    "min_samples_split":[2,5,10,20]
}

In [33]:
grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

In [16]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7, 8, 10],
                         'min_samples_split': [2, 5, 10, 20]},
             scoring='f1')

In [17]:
print(grid.best_params_)

{'max_depth': 3, 'min_samples_split': 2}


In [18]:
best_model = grid.best_estimator_

best_pred = best_model.predict(X_test)

In [19]:
print(classification_report(y_test, best_pred))

              precision    recall  f1-score   support

           0       0.80      0.88      0.84       105
           1       0.80      0.69      0.74        74

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [20]:
tuned_accuracy = accuracy_score(y_test, best_pred)

print(tuned_accuracy)

0.7988826815642458


In [21]:
comparison = pd.DataFrame({

"Model":[
"Original",
"Tuned"
],

"Accuracy":[
accuracy,
tuned_accuracy
]

})

comparison

,Model,Accuracy
0,Original,0.720670
1,Tuned,0.798883


The tuned Decision Tree model achieved better performance than the original model by selecting optimal values for max_depth and min_samples_split.

Although the improvement in accuracy was modest, the Precision, Recall, and F1-score improved, making the model more reliable.

This demonstrates that hyperparameter tuning helps improve model performance instead of relying on default settings.